<a href="https://colab.research.google.com/github/davidlealo/practicos_sisrec_2026/blob/main/estudio_m%C3%A9tricas_sistemas_recomendacion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 1. Métricas de Error de Predicción (Ratings Explícitos)

Estas métricas evalúan qué tan cerca estuvo la "nota" que tu sistema predijo frente a la calificación real que el usuario le dio al ítem.

#### Root Mean Squared Error (RMSE)
Es el estándar de la industria para predecir ratings continuos. Al elevar el error al cuadrado, penaliza fuertemente las predicciones que están muy alejadas de la realidad.

$$RMSE = \sqrt{\frac{1}{N}\sum_{i=1}^{N}(\hat{y}_i - y_i)^2}$$

**Implementación en Python:**
```python
import numpy as np

def rmse(y_true, y_pred):
    """
    y_true: array con los ratings reales
    y_pred: array con los ratings predichos
    """
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    
    # Calculamos la raíz de la media de los errores al cuadrado
    return np.sqrt(np.mean((y_true - y_pred)**2))
```

---

### 2. Métricas de Clasificación y Ranking (Top-K)

Estas son fundamentales cuando el objetivo es presentar al usuario una lista con los mejores $K$ ítems, sin importar el rating exacto, sino el orden.

#### Precision@K y Recall@K
Miden qué proporción de los ítems recomendados eran relevantes (Precisión) y qué proporción del total de ítems relevantes logramos capturar en nuestra recomendación (Recall).

* $Precision@K = \frac{|\text{Relevantes} \cap \text{Recomendados en Top } K|}{K}$
* $Recall@K = \frac{|\text{Relevantes} \cap \text{Recomendados en Top } K|}{|\text{Total de Relevantes}|}$

**Implementación en Python:**
```python
def precision_recall_at_k(recommended_items, relevant_items, k=10):
    """
    recommended_items: lista de IDs recomendados ordenados por probabilidad (mayor a menor)
    relevant_items: set o lista de IDs que el usuario realmente consumió
    """
    # Cortamos la lista a los top K
    top_k_recommended = recommended_items[:k]
    
    # Intersección: cuántos de los recomendados eran realmente relevantes
    hits = set(top_k_recommended).intersection(set(relevant_items))
    
    precision = len(hits) / k
    
    # Prevenir división por cero si el usuario en la prueba no tiene ítems relevantes
    recall = len(hits) / len(relevant_items) if len(relevant_items) > 0 else 0.0
    
    return precision, recall
```

#### Normalized Discounted Cumulative Gain (NDCG@K)
Esta es una métrica crucial para evaluar rankings puros. A diferencia de Precision y Recall, el NDCG asume que los aciertos en las primeras posiciones valen mucho más que los aciertos al final de la lista.

El Discounted Cumulative Gain (DCG) se calcula sumando la relevancia de cada ítem, pero "descontada" logarítmicamente según su posición $i$:

$$DCG_K = \sum_{i=1}^{K} \frac{rel_i}{\log_2(i + 1)}$$

El NDCG es simplemente el DCG dividido por el DCG Ideal (IDCG), que es el puntaje máximo posible si el modelo hubiera ordenado los ítems perfectamente de mayor a menor relevancia.

**Implementación en Python:**
```python
def dcg_at_k(relevances, k):
    """
    relevances: array de relevancias (ej. [1, 0, 1, 0, 0]) en el orden predicho por el modelo
    """
    relevances = np.asfarray(relevances)[:k]
    if relevances.size == 0:
        return 0.0
        
    # log2(i+1) donde i es el índice. En Python el índice empieza en 0,
    # por lo que el denominador correcto es log2(i+2)
    discounts = np.log2(np.arange(2, relevances.size + 2))
    return np.sum(relevances / discounts)

def ndcg_at_k(relevances, k):
    # 1. Calcular el DCG de tu modelo
    dcg = dcg_at_k(relevances, k)
    
    # 2. Calcular el DCG ideal (ordenando los aciertos primero)
    ideal_relevances = np.sort(relevances)[::-1]
    idcg = dcg_at_k(ideal_relevances, k)
    
    if idcg == 0.0:
        return 0.0
        
    # 3. Normalizar
    return dcg / idcg
```




$RMSE = \sqrt{\frac{1}{N}\sum_{i=1}^{N}(\hat{y}_i - y_i)^2}$

In [1]:
import numpy as np

def rmse(y_true, y_pred):
  y_true = np.array(y_true)
  y_pred = np.array(y_pred)

  return np.sqrt(np.mean((y_true - y_pred)**2))

In [2]:
y_real = [5, 3, 4, 4]
y_predicho = [4, 3, 4, 5]

# Desglose mental:
# 5 vs 4 -> error de 1 -> al cuadrado es 1
# 3 vs 3 -> error de 0 -> al cuadrado es 0
# 4 vs 4 -> error de 0 -> al cuadrado es 0
# 4 vs 5 -> error de -1 -> al cuadrado es 1
# Promedio de los cuadrados: (1 + 0 + 0 + 1) / 4 = 0.5
# Raíz cuadrada de 0.5 = 0.7071...

error = rmse(y_real, y_predicho)
print(f"El RMSE es: {error:.4f}") # Debería imprimir 0.7071

El RMSE es: 0.7071


¡Perfecto! Ese formato con `$` es el ideal para que tus apuntes de Markdown se vean impecables y profesionales.

Vamos con otra métrica clásica de la misma familia (predicción de ratings), ideal para seguir soltando la mano con la vectorización en `numpy`.

### Mean Absolute Error (MAE)

A diferencia del RMSE que castiga fuertemente los errores grandes (porque eleva la diferencia al cuadrado), el MAE simplemente nos da el promedio directo de las equivocaciones. Es muy útil cuando queremos saber el margen de error "típico" de nuestro modelo en la misma escala de las notas (por ejemplo, poder decirle al negocio: *"nuestro sistema se equivoca por 0.8 estrellas en promedio"*).

La fórmula matemática es:

$MAE = \frac{1}{N}\sum_{i=1}^{N}|\hat{y}_i - y_i|$

**Donde:**
* $N$ es el número total de predicciones.
* $\hat{y}_i$ es el rating predicho por tu sistema.
* $y_i$ es el rating real que dio el usuario.
* $| ... |$ representa el valor absoluto (para que los errores negativos y positivos no se cancelen entre sí).

**Tu misión:**
Escribe la función `def mae(y_true, y_pred):` asumiendo que vas a recibir listas estándar de Python y necesitas hacer el cálculo usando las bondades de `numpy`.

¡Pega tu código aquí cuando lo tengas y lo revisamos!

In [3]:
def mae(y_true, y_pred):
  y_true = np.array(y_true)
  y_pred = np.array(y_pred)

  return np.mean(np.abs(y_true - y_pred))

In [4]:
y_real = [5, 3, 4, 4]
y_predicho = [4, 3, 4, 5]

# Desglose mental para el MAE:
# 5 vs 4 -> error de 1 -> absoluto es 1
# 3 vs 3 -> error de 0 -> absoluto es 0
# 4 vs 4 -> error de 0 -> absoluto es 0
# 4 vs 5 -> error de -1 -> absoluto es 1
# Promedio de los absolutos: (1 + 0 + 0 + 1) / 4 = 0.5

error = mae(y_real, y_predicho)
print(f"El MAE es: {error}") # Imprime 0.5

El MAE es: 0.5


¡Excelente cambio de switch! Pasamos de evaluar "qué nota exacta le pondrá el usuario" a evaluar "¿le achuntamos o no con la lista de recomendaciones?". En el mundo real (y en tu doctorado), estas son las métricas que más le importan a los equipos de producto.

Aquí el concepto de vectorización matemática cambia un poco, porque ya no estamos restando números, sino que estamos haciendo **teoría de conjuntos** (intersecciones) y conteos.

### La lógica detrás del "Top K"

En los sistemas reales, no le muestras al usuario los 10.000 ítems de tu catálogo, le muestras una lista corta (un carrusel de Netflix, por ejemplo). Ese tamaño de la lista es **K**.

* **Precision@K:** De los $K$ ítems que le mostramos, ¿qué porcentaje le interesó realmente? (Mide la "calidad" de tu lista).
* **Recall@K:** De *todos* los ítems que sabemos que al usuario le interesan en el catálogo completo, ¿qué porcentaje logramos atrapar y meter dentro de ese Top $K$? (Mide tu "cobertura").

### La Matemática

$Precision@K = \frac{|\text{Relevantes} \cap \text{Recomendados en Top } K|}{K}$

$Recall@K = \frac{|\text{Relevantes} \cap \text{Recomendados en Top } K|}{|\text{Total de Relevantes}|}$

*(Nota: Los palitos $|...|$ aquí significan "cantidad de elementos", no valor absoluto).*

### Tu misión

Vas a escribir la función `def precision_recall_at_k(recomendados, relevantes, k):` que retorne ambos valores.

**Las reglas del juego para la prueba:**
1.  Recibes `recomendados`: Un arreglo de NumPy ordenado de mayor a menor probabilidad (el primer ítem es tu mejor recomendación).
2.  Recibes `relevantes`: Un arreglo de NumPy con los ítems que sabemos que al usuario le gustaron (el "ground truth").
3.  Corta la lista de `recomendados` hasta el índice $k$.
4.  Encuentra cuántos elementos se cruzan (intersección) entre ese Top $K$ y los `relevantes`. En NumPy, la función salvavidas para esto es `np.intersect1d(arreglo1, arreglo2)`.
5.  Calcula ambas métricas. **Ojo con una trampa clásica de las pruebas:** ¿Qué pasa si el arreglo `relevantes` viene vacío porque el usuario es nuevo y no tiene historial? ¡Evita que tu código explote por división por cero en el Recall!

Aquí tienes los datos de prueba para que los uses en tu código:

```python
import numpy as np

# IDs de los ítems
recomendados = np.array([101, 204, 309, 411, 599])
relevantes = np.array([204, 599, 888, 999])
k = 3

# Desglose mental para K=3:
# Tu Top 3 es: [101, 204, 309]
# Los relevantes son: [204, 599, 888, 999]
# La intersección en el Top 3 es solo el ítem 204 (1 acierto).
# Precision@3 = 1 / 3 = 0.333...
# Recall@3 = 1 / 4 = 0.25
```

¡Escribe tu código, pégalo aquí y lo revisamos a fondo!

**Voy a hacer primero**
$Precision@K = \frac{|\text{Relevantes} \cap \text{Recomendados en Top } K|}{K}$


In [6]:
def precision_at_k(recomendados, relevantes, k):
    # Cortamos la lista al Top K
    top_k = recomendados[:k]

    # Buscamos cuántos aciertos tuvimos
    interseccion = np.intersect1d(top_k, relevantes)

    # Calculamos la precisión
    return len(interseccion) / k

In [7]:
import numpy as np

# IDs de los ítems
recomendados = np.array([101, 204, 309, 411, 599])
relevantes = np.array([204, 599, 888, 999])
k = 3

# Desglose mental para K=3:
# Tu Top 3 es: [101, 204, 309]
# Los relevantes son: [204, 599, 888, 999]
# La intersección en el Top 3 es solo el ítem 204 (1 acierto).
# Precision@3 = 1 / 3 = 0.333...
# Recall@3 = 1 / 4 = 0.25

In [8]:
precision_at_k(recomendados, relevantes, k)

0.3333333333333333

**Voy a hacer segundo**
$Recall@K = \frac{|\text{Relevantes} \cap \text{Recomendados en Top } K|}{|\text{Total de Relevantes}|}$


In [9]:
def recall_at_k(relevantes, recomendados, k):
  top_k = recomendados[:k]
  interseccion = np.intersect1d(top_k, relevantes)

  if len(relevantes) == 0:
    return 0.0

  return len(interseccion) / len(relevantes)

In [10]:
recall_at_k(relevantes, recomendados, k)

0.25

¡Vamos por ese jefe final! El **NDCG@K** (Normalized Discounted Cumulative Gain) impone respeto porque penaliza el orden.

En Precision y Recall, te daba lo mismo si el ítem relevante estaba en el primer lugar o en el último del Top K; valían lo mismo. El NDCG dice: *"No, si el usuario tiene que hacer scroll para encontrar lo que le gusta, tu modelo es peor"*.

Para lograr esto, la métrica usa un logaritmo en el denominador para "descontar" el puntaje a medida que bajamos en la lista. Se calcula en 3 pasos:

### 1. El DCG (Discounted Cumulative Gain)
Es la suma de los aciertos, pero dividiendo cada acierto por el logaritmo de su posición.

$$DCG@K = \sum_{i=1}^{K} \frac{rel_i}{\log_2(i + 1)}$$

* $rel_i$: Es la relevancia del ítem en la posición $i$. En tu caso, será `1` si el ítem recomendado está en la lista de relevantes, y `0` si no.
* $i$: Es la posición en el ranking (1, 2, 3... hasta K). *Nota: le sumamos 1 al índice porque $\log_2(1) = 0$ y no queremos dividir por cero en la primera posición.*

### 2. El IDCG (Ideal DCG)
Es el máximo puntaje posible. Se calcula tomando la misma cantidad de aciertos que tuviste, pero imaginando que tu modelo fue perfecto y **los puso todos en los primeros lugares**.

$$IDCG@K = \sum_{i=1}^{K} \frac{rel_{ideal\_i}}{\log_2(i + 1)}$$

### 3. El NDCG (La normalización)
Para que la nota final sea siempre un número entre 0 y 1, simplemente divides tu puntaje por el puntaje perfecto:

$$NDCG@K = \frac{DCG@K}{IDCG@K}$$

Antes de pasar al código, juega un segundo con esta herramienta para que veas visualmente cómo el logaritmo "castiga" los aciertos tardíos frente a los aciertos en las primeras posiciones:

```json?chameleon
{"component":"LlmGeneratedComponent","props":{"height":"600px","prompt":"Objective: Create an interactive visualizer for NDCG@K (Normalized Discounted Cumulative Gain). Data State: K = 5 positions. Strategy: Form Layout. Inputs: 5 switches or checkboxes representing the 5 recommended ranking positions (Posición 1 to Posición 5). The user toggles if the item at that position is 'Relevante' (1) or 'No Relevante' (0). Behavior: Real-time calculation of DCG, IDCG, and NDCG based on the toggles. Display a visual breakdown showing the math for each active position: 1 / log2(position + 1). For example, show that position 1 yields 1.0 points, position 2 yields 0.63 points, position 3 yields 0.5 points, etc. Clearly show how the Ideal DCG (IDCG) rearranges the active 'Relevante' items to the very top positions to calculate the maximum possible score, and finally divide DCG by IDCG for the NDCG result. All text, titles, and labels must be in Spanish.","id":"im_20e5e0a606a426a4"}}
```

### Tu misión en Python

Escribir esta métrica requiere un paso previo: transformar tus listas de IDs en una matriz de `1`s y `0`s (relevancias).
El truco de magia en `numpy` para hacer esto sin usar un ciclo `for` es `np.isin()`.

```python
recomendados = np.array([101, 204, 309, 411])
relevantes = np.array([204, 411])

# Esto verifica qué recomendados están en la lista de relevantes
# y .astype(float) transforma los True/False en 1.0 y 0.0
relevancias = np.isin(recomendados, relevantes).astype(float)

print(relevancias)
# Resultado: [0., 1., 0., 1.] -> Fallaste el 1ro, achuntaste el 2do, fallaste el 3ro, achuntaste el 4to.
```

**Con ese truco bajo la manga, te toca programar:**
Intenta escribir la función `def ndcg_at_k(recomendados, relevantes, k):`.
*Pista:* `np.arange(2, k + 2)` te sirve para generar los números del denominador del logaritmo de una sola vez, y `np.log2()` te saca el logaritmo de todo el arreglo.

¡Pega tu intento y lo depuramos!

In [11]:
recomendados = np.array([101, 204, 309, 411])
relevantes = np.array([204, 411])

# Esto verifica qué recomendados están en la lista de relevantes
# y .astype(float) transforma los True/False en 1.0 y 0.0
relevancias = np.isin(recomendados, relevantes).astype(float)

print(relevancias)
# Resultado: [0., 1., 0., 1.] -> Fallaste el 1ro, achuntaste el 2do, fallaste el 3ro, achuntaste el 4to.

[0. 1. 0. 1.]


In [14]:
import numpy as np

def dcg_at_k(relevances, k):
    """
    relevances: array de relevancias (ej. [1, 0, 1, 0, 0])
    """
    # ¡Aquí está la corrección clave para NumPy 2.0+!
    relevances = np.asarray(relevances, dtype=float)[:k]

    if relevances.size == 0:
        return 0.0

    discounts = np.log2(np.arange(2, relevances.size + 2))
    return np.sum(relevances / discounts)

def ndcg_at_k(relevances, k):
    # 1. Calcular el DCG de tu modelo
    dcg = dcg_at_k(relevances, k)

    # 2. Calcular el DCG ideal (ordenando los aciertos primero)
    ideal_relevances = np.sort(relevances)[::-1]
    idcg = dcg_at_k(ideal_relevances, k)

    if idcg == 0.0:
        return 0.0

    # 3. Normalizar
    return dcg / idcg

In [15]:
dcg_at_k(relevantes, 2)

np.float64(463.31212871786903)

In [16]:
ndcg_at_k(relevantes, 2)

np.float64(0.8584469664048655)

Es completamente normal y es un síntoma de los tiempos que corren. Los asistentes de código como el de Colab o Copilot son herramientas fantásticas para la productividad, pero actúan como "rueditas de entrenamiento". Cuando te las quitan para una prueba, es natural sentir vértigo porque falta la memoria muscular de la sintaxis.

La mejor forma de soltar la mano y ganar confianza pura es hacer "gimnasia" de arreglos sin bucles `for`. Vamos a hacer un circuito de entrenamiento. Iremos de menos a más, enfocados en la lógica que se usa por debajo en los sistemas de recomendación.

### Nivel 1: Filtrado y Máscaras Booleanas

En los sistemas de recomendación, a menudo necesitamos convertir ratings explícitos (1 a 5 estrellas) en interacciones binarias (relevante o no relevante). Por ejemplo, podríamos decidir que solo las películas con nota 4 o 5 son "relevantes" para el usuario.

**El Desafío:**
Escribe una función que reciba un arreglo de notas y un valor de corte (umbral). La función debe devolver un nuevo arreglo **solo con los números que sean mayores o iguales a ese umbral**.

**Regla de oro:** ¡Cero ciclos `for`! Tienes que usar la lógica de filtrado directo de NumPy (a veces llamado *boolean indexing* o máscaras).

**Tu punto de partida:**
```python
import numpy as np

def filtrar_relevantes(ratings, umbral):
    ratings = np.array(ratings)
    
    # Escribe tu lógica aquí usando NumPy
    resultado = ...
    
    return resultado

# Datos de prueba:
mis_notas = [3, 5, 2, 4, 1, 5, 3]
corte = 4

# Si lo haces bien, print(filtrar_relevantes(mis_notas, corte))
# debería devolver exactamente el arreglo: [5 4 5]
```

¿Te animas a escribir la línea de código que falta para resolver este primer nivel? ¡Pégala aquí y avanzamos al siguiente!

In [23]:
import numpy as np

def filtrar_relevantes(ratings, umbral):
    ratings = np.array(ratings)

    # Escribe tu lógica aquí usando NumPy
    resultado = ratings[ratings >= umbral]

    return resultado

# Datos de prueba:
mis_notas = [3, 5, 2, 4, 1, 5, 3]
corte = 4

# Si lo haces bien, print(filtrar_relevantes(mis_notas, corte))
# debería devolver exactamente el arreglo: [5 4 5]

filtrar_relevantes(mis_notas, corte)

array([5, 4, 5])

In [18]:
import numpy as np

ratings = np.array([3, 5, 2, 4, 1, 5, 3])
umbral = 4

mascara = ratings >= umbral
print(mascara)
# Imprime: [False, True, False, True, False, True, False]

[False  True False  True False  True False]


In [19]:
resultado = ratings[mascara]
print(resultado)
# Imprime: [5, 4, 5] -> ¡Misión cumplida!

[5 4 5]


In [20]:
def filtrar_relevantes(ratings, umbral):
    ratings = np.array(ratings)

    # La máscara booleana filtra el arreglo directamente
    return ratings[ratings >= umbral]

In [22]:
# Datos de prueba:
mis_notas = [3, 5, 2, 4, 1, 5, 3]
corte = 4

filtrar_relevantes(mis_notas, corte)

array([5, 4, 5])